# 03 · Train the visual model on LRS3 trainval (GPU)
Add inputs: the outputs of **lrs3-trainval-0/1/2** and **lrs3-test**. Accelerator **GPU T4**, Internet **On**.

Run it twice (one notebook copy each), changing `TRACK` in the first cell:
| copy | `TRACK` | model |
|---|---|---|
| visual-vit | `'vit'` | Track A: VideoMAE ViT-Base + temporal Transformer |
| visual-autoavsr | `'autoavsr'` | Track B: pre-trained Auto-AVSR encoder + new CTC heads |

The training saves `last.pt` every epoch. If the 12-hour session ends, add this notebook's own previous output as input, set `RESUME_FROM`, and run again. At the end it evaluates PER/VER and writes the confusion matrices (`eval_val/`, `eval_test/`).

In [ ]:
TRACK = 'vit'            # 'vit' (Track A) or 'autoavsr' (Track B)
RESUME_FROM = None       # e.g. '/kaggle/input/<previous-output>/runs/visual_lrs3_vit'

In [ ]:
# --- setup: clone the project code ---
import os, subprocess
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/durgaanuhya02/MoVi-Swin-Motion--and-Viseme-Aware-Space-Time-Swin-Transformer-for-Robust-and-Efficient-Lip-Reading.git", "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
!git log --oneline -1

In [ ]:
!pip install -q cmudict transformers==4.57.1 peft==0.21.2 gdown   # versions the models were tested with

In [ ]:
import glob, json
INDEX = sorted(glob.glob('/kaggle/input/**/lrs3_trainval_shard*/index.jsonl', recursive=True))
TEST_INDEX = glob.glob('/kaggle/input/**/lrs3_test_rois/index.jsonl', recursive=True)
INDEX_JSON = json.dumps(INDEX)
print(INDEX, TEST_INDEX)
CONFIG = f'configs/visual_lrs3_{TRACK}.yaml'
OUT = f'/kaggle/working/runs/visual_lrs3_{TRACK}'

### Track B only: Auto-AVSR code and checkpoint
If the Google Drive download fails (quota), download `vsr_trlrs3vox2_base.pth` from the [Auto-AVSR model zoo](https://github.com/mpc001/auto_avsr), upload it as a Kaggle dataset, add it as input and set `CKPT` to its path.

In [ ]:
CKPT = '/kaggle/working/vsr_trlrs3vox2_base.pth'
if TRACK == 'autoavsr':
    if not os.path.exists('/kaggle/working/auto_avsr'):
        !git clone -q https://github.com/mpc001/auto_avsr /kaggle/working/auto_avsr
        !cd /kaggle/working/auto_avsr && git checkout -q 182b62837773ab01052d4ac21ef1d2203ea7d267
    uploaded = glob.glob('/kaggle/input/**/vsr_trlrs3vox2_base.pth', recursive=True)
    if uploaded:
        CKPT = uploaded[0]
    elif not os.path.exists(CKPT):
        !gdown 1shcWXUK2iauRhW9NbwCc25FjU1CoMm8i -O {CKPT}
    !ls -lh {CKPT}

### Check: do our trainval crops look like the pre-cropped LRS3 test crops?
Top rows: our MediaPipe crops (trainval). Bottom rows: test-set crops. If the mouth size or position differs a lot, re-run notebook 01 with a different `--crop-scale` before training.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from src.data.dataset import ROIDataset
from src.preprocess.extract_rois import read_video
def frames_of(index, n):
    ds = ROIDataset(index)
    return [read_video(f"{e['_root']}/{e['roi']}")[10] for e in ds.entries[:n]]
row1 = frames_of(INDEX[0], 6); row2 = frames_of(TEST_INDEX[0], 6)
plt.figure(figsize=(12, 4)); plt.imshow(np.vstack([np.hstack(row1), np.hstack(row2)]), cmap='gray'); plt.axis('off')
plt.title('top: our trainval crops   |   bottom: LRS3 test crops'); plt.show()

In [ ]:
import shutil
os.makedirs(OUT, exist_ok=True)
if RESUME_FROM:
    for f in ['last.pt', 'best.pt', 'log.jsonl']:
        if os.path.exists(f'{RESUME_FROM}/{f}'): shutil.copy(f'{RESUME_FROM}/{f}', OUT)

In [ ]:
EXTRA = f'model.checkpoint={CKPT}' if TRACK == 'autoavsr' else ''
!python -m src.train_visual --config {CONFIG} --set 'index={INDEX_JSON}' out_dir={OUT} {EXTRA}

In [ ]:
!tail -5 {OUT}/log.jsonl

## PER / VER and confusion matrices (validation and test)

In [ ]:
!python -m src.eval_visual --checkpoint {OUT}/best.pt --index {' '.join(INDEX)} --split val --out {OUT}/eval_val
!python -m src.eval_visual --checkpoint {OUT}/best.pt --index {TEST_INDEX[0]} --split test --out {OUT}/eval_test

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUT}/eval_test/confusion_phoneme.png'))
display(Image(f'{OUT}/eval_test/confusion_viseme.png'))
!rm -f {OUT}/last.pt   # keep only best.pt in the saved output